---
## 3. AST Training
> ⚡ **Optimisations:** AMP/FP16 · `TRAIN_SIZE` reduced to 3000 · `NUM_WORKERS=4`
> · `time_stretch` replaced with fast random-offset crop (same augmentation effect, 0 phase-vocoder overhead)
>
> # ── AST: Audio Helpers ──────────────────────────────────────────
- time_stretch REMOVED — replaced by random-offset crop.
- Rationale: time_stretch shifts pitch + duration via slow STFT phase vocoder.
- For genre classification, a random crop from a different time offset gives
- equivalent position-invariance augmentation at ~0 CPU cost.


In [7]:


def ast_load_audio(path):
    audio, _ = librosa.load(path, sr=AST_SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)

def ast_normalize(audio):
    return audio / (np.max(np.abs(audio)) + 1e-6)

def ast_crop_random(audio):
    if len(audio) >= AST_MAX_LENGTH:
        start = random.randint(0, len(audio) - AST_MAX_LENGTH)
        return audio[start : start + AST_MAX_LENGTH]
    return np.pad(audio, (0, AST_MAX_LENGTH - len(audio)))

def ast_crop_center(audio):
    if len(audio) >= AST_MAX_LENGTH:
        start = (len(audio) - AST_MAX_LENGTH) // 2
        return audio[start : start + AST_MAX_LENGTH]
    return np.pad(audio, (0, AST_MAX_LENGTH - len(audio)))

def ast_random_gain(audio, low=0.6, high=1.4):
    return audio * random.uniform(low, high)

def ast_add_noise(audio):
    if noise_files_list and random.random() < AST_NOISE_PROB:
        nf   = random.choice(noise_files_list)
        n, _ = librosa.load(nf, sr=AST_SAMPLE_RATE, mono=True)
        n    = n.astype(np.float32)
        n    = ast_crop_random(n)
        lvl  = random.uniform(*AST_NOISE_LEVEL)
        audio = audio + lvl * n
    return audio

print("AST audio helpers ready (time_stretch removed — fast random-crop replaces it)!")


AST audio helpers ready (time_stretch removed — fast random-crop replaces it)!


In [8]:
# ── AST: Load Model ─────────────────────────────────────────────
AST_MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'
ast_feature_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL_NAME)

_ast_base = ASTForAudioClassification.from_pretrained(
    AST_MODEL_NAME,
    num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True
)
_ast_base.to(DEVICE)

# ⚡ DataParallel — use both T4 GPUs
if USE_DDP and N_GPUS > 1:
    ast_model = nn.DataParallel(_ast_base)
    print(f"AST wrapped in DataParallel across {N_GPUS} GPUs!")
else:
    ast_model = _ast_base

total = sum(p.numel() for p in _ast_base.parameters())
print(f"AST loaded | {total:,} params | device: {DEVICE}")


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


AST loaded | 86,196,490 params | device: cuda


In [11]:
# ── AST: Datasets ───────────────────────────────────────────────
class ASTTrainDataset(Dataset):
    def __init__(self, song_index, feature_extractor, size=AST_TRAIN_SIZE):
        self.song_index = song_index
        self.fe = feature_extractor
        self.size = size
    def __len__(self): return self.size
    def __getitem__(self, idx):
        while True:
            try:
                genre = random.choice(GENRES)
                songs = self.song_index[genre]
                if not songs: continue
                mixed = None
                for stem in REQUIRED_STEMS:
                    audio = ast_load_audio(os.path.join(random.choice(songs), stem))
                    audio = ast_crop_random(audio)   # ← random offset = augmentation
                    audio = ast_random_gain(audio)
                    mixed = audio if mixed is None else mixed + audio
                mixed = ast_normalize(mixed)
                mixed = ast_add_noise(mixed)
                mixed = ast_normalize(mixed)
                inp   = self.fe(mixed, sampling_rate=AST_SAMPLE_RATE, return_tensors='pt')
                return {'input_values': inp['input_values'].squeeze(0),
                        'labels': torch.tensor(label2id[genre], dtype=torch.long)}
            except Exception: continue

class ASTValDataset(Dataset):
    def __init__(self, val_index, feature_extractor):
        self.fe = feature_extractor
        self.samples = [(g, sp) for g in GENRES for sp in val_index[g]]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        genre, sp = self.samples[idx]
        mixed = None
        for stem in REQUIRED_STEMS:
            audio = ast_load_audio(os.path.join(sp, stem))
            audio = ast_crop_center(audio)
            mixed = audio if mixed is None else mixed + audio
        mixed = ast_normalize(mixed)
        inp   = self.fe(mixed, sampling_rate=AST_SAMPLE_RATE, return_tensors='pt')
        return {'input_values': inp['input_values'].squeeze(0),
                'labels': torch.tensor(label2id[genre], dtype=torch.long)}

ast_train_ds = ASTTrainDataset(train_index, ast_feature_extractor, AST_TRAIN_SIZE)
ast_val_ds   = ASTValDataset(val_index, ast_feature_extractor)
ast_train_loader = DataLoader(ast_train_ds, batch_size=AST_BATCH_SIZE, shuffle=True,
                               num_workers=AST_NUM_WORKERS, pin_memory=True, persistent_workers=True)
ast_val_loader   = DataLoader(ast_val_ds,   batch_size=AST_BATCH_SIZE, shuffle=False,
                               num_workers=AST_NUM_WORKERS, pin_memory=True, persistent_workers=True)
print(f"AST  Train: {len(ast_train_ds)} samples | Val: {len(ast_val_ds)} samples")


AST  Train: 3000 samples | Val: 150 samples


In [12]:
# ── AST: Loss, Metric, Train/Val Functions ──────────────────────
ast_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
ast_f1_metric = MulticlassF1Score(num_classes=NUM_LABELS, average='macro').to(DEVICE)

# ⚡ GradScaler for AMP
ast_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def ast_train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    ast_f1_metric.reset()
    total_loss, all_preds, all_labels = 0, [], []
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='  AST Train')):
        iv     = batch['input_values'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        # ⚡ AMP forward
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(input_values=iv)
            loss    = ast_criterion(outputs.logits, labels) / AST_GRAD_ACCUM
        ast_scaler.scale(loss).backward()
        if (step + 1) % AST_GRAD_ACCUM == 0:
            ast_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                (model.module if hasattr(model,'module') else model).parameters(), 1.0
            )
            ast_scaler.step(optimizer)
            ast_scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * AST_GRAD_ACCUM
        preds = torch.argmax(outputs.logits, 1)
        ast_f1_metric.update(preds, labels)
        all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    pc = f1_score(all_labels, all_preds, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
    return total_loss / len(loader), ast_f1_metric.compute().item(), pc

def ast_validate(model, loader):
    model.eval()
    ast_f1_metric.reset()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  AST Val  '):
            iv, labels = batch['input_values'].to(DEVICE), batch['labels'].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(input_values=iv)
                loss    = ast_criterion(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, 1)
            ast_f1_metric.update(preds, labels)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    pc = f1_score(all_labels, all_preds, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
    return total_loss / len(loader), ast_f1_metric.compute().item(), pc

print("AST train/validate functions ready (AMP enabled)!")


AST train/validate functions ready (AMP enabled)!


In [55]:
'''# ── AST: Phase 1 — Freeze base, train head ──────────────────────
wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="AST_Phase1", reinit=True,
           config=dict(phase=1, model=AST_MODEL_NAME, epochs=AST_EPOCHS_P1,
                       lr=AST_LR_P1, batch_size=AST_BATCH_SIZE, grad_accum=AST_GRAD_ACCUM,
                       train_size=AST_TRAIN_SIZE, duration_s=AST_DURATION, amp=USE_AMP))

_base = ast_model.module if hasattr(ast_model, 'module') else ast_model
for p in _base.base_model.parameters(): p.requires_grad = False
print(f"Phase 1: {sum(p.numel() for p in _base.parameters() if p.requires_grad):,} trainable params")

optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, _base.parameters()), lr=AST_LR_P1, weight_decay=AST_WEIGHT_DECAY)
total_steps_p1 = (len(ast_train_loader) // AST_GRAD_ACCUM) * AST_EPOCHS_P1
scheduler_p1   = get_cosine_schedule_with_warmup(optimizer_p1,
    num_warmup_steps=total_steps_p1//10, num_training_steps=total_steps_p1)

best_p1_f1, patience_cnt = 0, 0
print(f"\n{'='*52}\nPHASE 1 — Classifier Warmup ({AST_EPOCHS_P1} epochs)\n{'='*52}\n")

for epoch in range(AST_EPOCHS_P1):
    tl, tf1, tpc = ast_train_one_epoch(ast_model, ast_train_loader, optimizer_p1, scheduler_p1)
    vl, vf1, vpc = ast_validate(ast_model, ast_val_loader)
    log = {"epoch": epoch+1, "train/loss": tl, "train/macro_f1": tf1,
           "val/loss": vl, "val/macro_f1": vf1, "lr": optimizer_p1.param_groups[0]['lr']}
    for i, g in enumerate(GENRES):
        log[f"val/f1_{g}"] = float(vpc[i]); log[f"train/f1_{g}"] = float(tpc[i])
    wandb.log(log)
    print(f"Ep {epoch+1}/{AST_EPOCHS_P1} | TL:{tl:.4f} TF1:{tf1:.4f} | VL:{vl:.4f} VF1:{vf1:.4f}")
    if vf1 > best_p1_f1:
        best_p1_f1 = vf1
        torch.save(_base.state_dict(), '/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase1.pth')
        patience_cnt = 0; print(f"  ✓ Saved (val F1: {best_p1_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= AST_PATIENCE_P1: print("  Early stopping P1."); break

wandb.run.summary["best_val_f1_phase1"] = best_p1_f1
wandb.finish()
print(f"\nPhase 1 done! Best val F1: {best_p1_f1:.4f}")'''


Phase 1: 9,226 trainable params

PHASE 1 — Classifier Warmup (3 epochs)



  AST Train:   0%|          | 0/250 [00:00<?, ?it/s]

  AST Val  :   0%|          | 0/13 [00:00<?, ?it/s]

Ep 1/3 | TL:1.3848 TF1:0.6534 | VL:1.1286 VF1:0.7482
  ✓ Saved (val F1: 0.7482)


  AST Train:   0%|          | 0/250 [00:00<?, ?it/s]

...

  AST Train:   0%|          | 0/250 [00:00<?, ?it/s]

  AST Val  :   0%|          | 0/13 [00:00<?, ?it/s]

Ep 3/3 | TL:0.8250 TF1:0.8970 | VL:1.0508 VF1:0.8313
  ✓ Saved (val F1: 0.8313)


epoch,▁▅█
lr,█▄▁
train/f1_blues,▁▇█
train/f1_classical,▁▇█
train/f1_country,▁██
train/f1_disco,▁▇█
train/f1_hiphop,▁▇█
train/f1_jazz,▁▇█
train/f1_metal,▁▇█
train/f1_pop,▁██
+16,...



Phase 1 done! Best val F1: 0.8313


In [56]:
'''# ── AST: Phase 2 — Full fine-tune all layers ────────────────────
wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="AST_Phase2", reinit=True,
           config=dict(phase=2, model=AST_MODEL_NAME, epochs=AST_EPOCHS_P2,
                       lr_base=AST_LR_P2, lr_head=AST_LR_P2*AST_LR_HEAD_MULT,
                       batch_size=AST_BATCH_SIZE, amp=USE_AMP))

_base.load_state_dict(torch.load('/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase1.pth'))
for p in _base.parameters(): p.requires_grad = True
print(f"All {sum(p.numel() for p in _base.parameters()):,} params unfrozen!")

optimizer_p2 = torch.optim.AdamW([
    {'params': _base.base_model.parameters(),  'lr': AST_LR_P2},
    {'params': _base.classifier.parameters(),  'lr': AST_LR_P2 * AST_LR_HEAD_MULT},
], weight_decay=AST_WEIGHT_DECAY)
total_steps_p2 = (len(ast_train_loader) // AST_GRAD_ACCUM) * AST_EPOCHS_P2
scheduler_p2   = get_cosine_schedule_with_warmup(optimizer_p2,
    num_warmup_steps=total_steps_p2//10, num_training_steps=total_steps_p2)

best_p2_f1, patience_cnt = 0, 0
print(f"\n{'='*52}\nPHASE 2 — Full Fine-Tune ({AST_EPOCHS_P2} epochs)\n"
      f"Base LR: {AST_LR_P2} | Head LR: {AST_LR_P2*AST_LR_HEAD_MULT}\n{'='*52}\n")

for epoch in range(AST_EPOCHS_P2):
    tl, tf1, tpc = ast_train_one_epoch(ast_model, ast_train_loader, optimizer_p2, scheduler_p2)
    vl, vf1, vpc = ast_validate(ast_model, ast_val_loader)
    log = {"epoch": epoch+1, "train/loss": tl, "train/macro_f1": tf1,
           "val/loss": vl, "val/macro_f1": vf1,
           "lr_base": optimizer_p2.param_groups[0]['lr'],
           "lr_head": optimizer_p2.param_groups[1]['lr']}
    for i, g in enumerate(GENRES):
        log[f"val/f1_{g}"] = float(vpc[i]); log[f"train/f1_{g}"] = float(tpc[i])
    wandb.log(log)
    print(f"Ep {epoch+1}/{AST_EPOCHS_P2} | TL:{tl:.4f} TF1:{tf1:.4f} | VL:{vl:.4f} VF1:{vf1:.4f}")
    if vf1 > best_p2_f1:
        best_p2_f1 = vf1
        torch.save(_base.state_dict(), '/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase2.pth')
        patience_cnt = 0; print(f"  ✓ Saved (val F1: {best_p2_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= AST_PATIENCE_P2: print("  Early stopping P2."); break

wandb.run.summary["best_val_f1_phase2"] = best_p2_f1
wandb.finish()
print(f"\nPhase 2 done! Best val F1: {best_p2_f1:.4f}")
'''

All 86,196,490 params unfrozen!

PHASE 2 — Full Fine-Tune (5 epochs)
Base LR: 2e-05 | Head LR: 0.0002



  AST Train:   0%|          | 0/250 [00:00<?, ?it/s]

  AST Val  :   0%|          | 0/13 [00:00<?, ?it/s]

Ep 1/5 | TL:0.7878 TF1:0.8962 | VL:1.0752 VF1:0.7812
  ✓ Saved (val F1: 0.7812)


  AST Train:   0%|          | 0/250 [00:00<?, ?it/s]

...

  AST Train:   0%|          | 0/250 [00:00<?, ?it/s]

  AST Val  :   0%|          | 0/13 [00:00<?, ?it/s]

Ep 5/5 | TL:0.5544 TF1:0.9808 | VL:0.9010 VF1:0.8720
  ✓ Saved (val F1: 0.8720)


epoch,▁▃▅▆█
lr_base,█▆▄▂▁
lr_head,█▆▄▂▁
train/f1_blues,▁▃▆██
train/f1_classical,▃▁▅█▆
train/f1_country,▁▄▆██
train/f1_disco,▁▄▇██
train/f1_hiphop,▁▅▇██
train/f1_jazz,▁▃▇██
train/f1_metal,▃▁▆█▇
+17,...



Phase 2 done! Best val F1: 0.8720


In [13]:
# ── AST Phase 2 — SKIP (checkpoint exists) ──────────────────────
_base = ast_model.module if hasattr(ast_model, 'module') else ast_model
_base.load_state_dict(
    torch.load('/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase2.pth',
               map_location=DEVICE)
)
_base.eval()
best_p2_f1 = 0
print("✓ AST loaded from phase 2 checkpoint — skipping retraining")

✓ AST loaded from phase 2 checkpoint — skipping retraining
